# SQLForge — 02: LoRA SFT (Colab, T4)

QLoRA fine-tune of Qwen2.5-Coder-1.5B-Instruct on the processed Spider train set.
Roughly 2h on a T4 for 2 epochs. **Resumable:** if Colab disconnects, re-run all
cells — training picks up from the last checkpoint in Drive.

In [ ]:
REPO_URL = "https://github.com/Adityakumarjha087/SQLFORGE-LLM.git"
from google.colab import drive
drive.mount('/content/drive')
!git clone $REPO_URL sqlforge
%cd sqlforge
!pip install -q uv && uv sync --group ml --group train
# Checkpoints live on Drive so disconnects lose nothing.
!mkdir -p /content/drive/MyDrive/sqlforge/outputs/sft && ln -sfn /content/drive/MyDrive/sqlforge/outputs outputs

In [ ]:
!uv run python scripts/download_spider.py
!uv run python scripts/build_dataset.py

In [ ]:
!uv run python training/sft.py

In [ ]:
# Loss curves
import json, matplotlib.pyplot as plt
logs = json.load(open('outputs/sft/log_history.json'))
train = [(l['step'], l['loss']) for l in logs if 'loss' in l]
val = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]
plt.plot(*zip(*train), label='train')
if val: plt.plot(*zip(*val), label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('SFT loss')
plt.savefig('docs/assets/sft_loss.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Evaluate the SFT adapter on the full test set.
!uv run python scripts/run_eval.py --name sft-full --adapter outputs/sft/adapter --batch-size 16 --log-results